In [ ]:
from conformal_generation.calibration_dataset import IndividualScoreCalibrationDataset
from conformal_generation.sequence_selector import RunningMaxSingleSequenceSelector
from conformal_generation.conformal_generation import ConformalGeneration
from conformal_generation.utils import plot_admissibility_curve, plot_generation_sequence_length

import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# PATH TO DATA FILE
# NOTE: the file below does not contain the opriginal images nor the corresponding generations to keep the file small and shareable.
# Check the README if you want access to the full data
memorization_file_path = "./data/conformal_memorization_prevention_data_no_images.pkl"
df = pd.read_pickle(memorization_file_path)

In [ ]:
df

In [ ]:
def score_fn(x, y):
    df_local = df[(df["original_im_num"]==x) & (df["gen_im_t"]==y)]
    assert df_local.shape[0] == 1
    return df_local["score"].iloc[0]
    
sequence_selector = RunningMaxSingleSequenceSelector(score_fn=score_fn)

In [ ]:
n = 26
np.random.seed(42)

df = df[df["gen_im_t"] <= 20]

calibration_im_nums = np.random.choice(df["original_im_num"].unique().tolist(), size=n, replace=False).tolist()
df_cal = df[df["original_im_num"].isin(calibration_im_nums)]
df_test = df[~df["original_im_num"].isin(calibration_im_nums)]
x_cal = df_cal["original_im_num"].unique().tolist()
x_test = df_test["original_im_num"].unique().tolist()
y_cal = df_cal.groupby("original_im_num", sort=False)["gen_im_t"].apply(list).tolist()
y_test = df_test.groupby("original_im_num", sort=False)["gen_im_t"].apply(list).tolist()

calibration_dataset = IndividualScoreCalibrationDataset(
    sequence_selector = sequence_selector,
    input_dataset = x_cal,
    raw_generated_dataset = y_cal,
    admissibility_dataset = df_cal.groupby("original_im_num", sort=False)["not_memorized"].apply(list).tolist(),
    admissibility_aggregation = lambda x: max(x)
)

CG = ConformalGeneration(sequence_selector=sequence_selector, calibration_dataset=calibration_dataset)

In [ ]:
def get_too_different(x, y):
    df_local = df[(df["original_im_num"]==x) & (df["gen_im_t"]==y)]
    assert df_local.shape[0] == 1
    return df_local["too_different"].iloc[0]

def get_admissibility(x, y):
    df_local = df[(df["original_im_num"]==x) & (df["gen_im_t"]==y)]
    assert df_local.shape[0] == 1
    return df_local["not_memorized"].iloc[0]

In [ ]:
gamma_list = np.linspace(0, 1, 21).tolist()
results = {gamma: [] for gamma in gamma_list}
is_lambda_inf = {gamma: False for gamma in gamma_list}
too_different = {gamma: [] for gamma in gamma_list}
for gamma in gamma_list:
    CG.calibrate(gamma=gamma, recalibrate=True)
    if CG.conformal_threshold == math.inf:
        is_lambda_inf[gamma] = True
        results[gamma] = [1 for _ in range(len(x_test))]
        for i in range(len(x_test)):
            too_different[gamma].append(get_too_different(x_test[i], y_test[i][-1]))
        continue
    for i in range(len(x_test)):
        selected = CG.select(instance=x_test[i], raw_generated_sequence=y_test[i])
        results[gamma].append(get_admissibility(x_test[i], selected[0]))
        too_different[gamma].append(get_too_different(x_test[i], selected[0]))
        
est = [np.mean(results[gamma]) for gamma in gamma_list]
diff = [np.mean(too_different[gamma]) for gamma in gamma_list]
admissibility_results = dict(zip(gamma_list, est))
too_diff_results = dict(zip(gamma_list, diff))

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16, 6), constrained_layout=False)
plot_admissibility_curve(ax1, admissibility_results, is_lambda_inf, is_calibration=False)
plot_generation_sequence_length(ax2, too_diff_results, y_label="% of Unrelated Images", plot_diagonal=True)
# plt.savefig("image_mem_fig.pdf", format="pdf",bbox_inches='tight')
plt.show()